In [44]:
# ===============================================================================
# POST-ESTIMATION ANALYSIS - IMPORT RESULTS FROM NOTEBOOK 6
# ===============================================================================

import pandas as pd
import numpy as np
import pickle
import matplotlib.pyplot as plt
import seaborn as sns

print("📥 LOADING RESULTS FROM NOTEBOOK 6 FOR POST-ESTIMATION ANALYSIS")
print("="*70)

# Load demand model results
with open('demand_model_results.pkl', 'rb') as f:
    demand_param = pickle.load(f)

# Load supply model results
with open('charging_station_model_results.pkl', 'rb') as f:
    charging_param = pickle.load(f)

# Load elasticity matrices (without network effects)
with open('elasticity_matrices.pkl', 'rb') as f:
    elasticity_matrices = pickle.load(f)

# Load total elasticity matrices (with network effects)
with open('total_elasticity_matrices.pkl', 'rb') as f:
    total_elasticity_matrices = pickle.load(f)

# Load gamma dictionary (charging station semi-elasticities)
with open('gamma_dict.pkl', 'rb') as f:
    gamma_dict = pickle.load(f)

# Load data for analysis
df = pd.read_csv('demand_counterfactual.csv')
supply_df = pd.read_csv('supply_counterfactual.csv')

# Add EV_share column if not present
if 'EV_share' not in df.columns:
    df['EV_share'] = df.groupby('market_ids')['is_electric'].transform(lambda x: x.mean())

print("✅ Successfully loaded:")
print(f"  • Demand model parameters: {len(demand_param.params)} coefficients")
print(f"  • Supply model parameters: {len(charging_param.params)} coefficients") 
print(f"  • Elasticity matrices: {len(elasticity_matrices)} markets (without network effects)")
print(f"  • Total elasticity matrices: {len(total_elasticity_matrices)} markets (with network effects)")
print(f"  • Gamma dictionary: {len(gamma_dict)} markets (charging station semi-elasticities)")
print(f"  • Demand data: {len(df)} observations across {df['market_ids'].nunique()} markets")
print(f"  • Supply data: {len(supply_df)} observations")
print()

# Create total_derivatives_dict for backward compatibility with existing code
total_derivatives_dict = {}
for market_id, matrix in total_elasticity_matrices.items():
    total_derivatives_dict[market_id] = matrix.values

# Create derivative_matrices if needed for the sample table generation
derivative_matrices = {}
for market_id in elasticity_matrices.keys():
    # For derivatives, we can use the elasticity matrix divided by prices
    # This is a simplified approach - adjust if you have actual derivative matrices
    derivative_matrices[market_id] = elasticity_matrices[market_id].copy()

print("🎯 Data ready for post-estimation analysis:")
print("  • Network effects analysis")
print("  • Market pattern identification") 
print("  • Sample table generation")
print("  • Feedback denominator analysis")

📥 LOADING RESULTS FROM NOTEBOOK 6 FOR POST-ESTIMATION ANALYSIS
✅ Successfully loaded:
  • Demand model parameters: 8 coefficients
  • Supply model parameters: 35 coefficients
  • Elasticity matrices: 155 markets (without network effects)
  • Total elasticity matrices: 155 markets (with network effects)
  • Gamma dictionary: 155 markets (charging station semi-elasticities)
  • Demand data: 33545 observations across 155 markets
  • Supply data: 155 observations

🎯 Data ready for post-estimation analysis:
  • Network effects analysis
  • Market pattern identification
  • Sample table generation
  • Feedback denominator analysis
✅ Successfully loaded:
  • Demand model parameters: 8 coefficients
  • Supply model parameters: 35 coefficients
  • Elasticity matrices: 155 markets (without network effects)
  • Total elasticity matrices: 155 markets (with network effects)
  • Gamma dictionary: 155 markets (charging station semi-elasticities)
  • Demand data: 33545 observations across 155 markets


In [45]:
# ===============================================================================
# FEEDBACK ANALYSIS - ANALYZE DENOMINATORS FOR NETWORK EFFECTS
# ===============================================================================

print("🔍 ANALYZING FEEDBACK DENOMINATORS ACROSS MARKETS")
print("="*60)

# Calculate feedback analysis for all markets
feedback_analysis = {'market_info': []}

beta_N = demand_param.params['log_charging_stock_hat'] + demand_param.params['is_electric:log_charging_stock_hat']
v_2 = charging_param.params['log(EV_stock)']

for market_id in df['market_ids'].unique():
    market_df = df[df['market_ids'] == market_id].copy()
    
    # Get EV share and gamma values for this market
    s_ev = market_df['EV_share'].iloc[0]
    gamma_values = gamma_dict[market_id]
    is_ev = market_df['is_electric'].values.astype(bool)
    
    # Calculate denominator: s_ev - v_2 * sum(gamma_ev)
    sum_gamma_ev = np.sum(gamma_values[is_ev])
    denominator = s_ev - v_2 * sum_gamma_ev
    
    # Count EVs and total products
    num_evs = market_df['is_electric'].sum()
    num_products = len(market_df)
    
    feedback_analysis['market_info'].append({
        'market_id': market_id,
        'denominator': denominator,
        's_ev': s_ev,
        'sum_gamma_ev': sum_gamma_ev,
        'num_evs': num_evs,
        'num_products': num_products
    })

print(f"✅ Analyzed {len(feedback_analysis['market_info'])} markets")

# Quick summary
positive_denom_count = sum(1 for info in feedback_analysis['market_info'] if info['denominator'] > 0)
print(f"   - Markets with positive denominators: {positive_denom_count}")
print(f"   - Markets with negative/zero denominators: {len(feedback_analysis['market_info']) - positive_denom_count}")

🔍 ANALYZING FEEDBACK DENOMINATORS ACROSS MARKETS
✅ Analyzed 155 markets
   - Markets with positive denominators: 93
   - Markets with negative/zero denominators: 62
✅ Analyzed 155 markets
   - Markets with positive denominators: 93
   - Markets with negative/zero denominators: 62


In [46]:
# ===============================================================================
# FIND MARKET WITH POSITIVE DENOMINATORS AND MODELS WITH NEGATIVE NETWORK EFFECTS
# ===============================================================================

print("🔍 ANALYZING MARKETS AND MODELS FOR SAMPLE SELECTION")
print("="*70)

# Get market information from feedback analysis
market_info = feedback_analysis['market_info']

# Find markets with positive denominators
positive_denom_markets = []
for info in market_info:
    if info['denominator'] > 0:
        positive_denom_markets.append({
            'market_id': info['market_id'],
            'denominator': info['denominator'],
            's_ev': info['s_ev'],
            'num_evs': info['num_evs'],
            'num_products': info['num_products']
        })

# Sort by denominator value (highest first for stability)
positive_denom_markets.sort(key=lambda x: x['denominator'], reverse=True)

print(f"📊 Found {len(positive_denom_markets)} markets with positive denominators")
print("\nTop 5 markets with highest positive denominators:")
for i, market in enumerate(positive_denom_markets[:5]):
    print(f"  {i+1}. {market['market_id']}: denominator={market['denominator']:.6f}, "
          f"s_ev={market['s_ev']:.6f}, EVs={market['num_evs']}, products={market['num_products']}")

# Select the market with highest positive denominator and sufficient EVs
selected_market = None
for market in positive_denom_markets:
    if market['num_evs'] >= 4:  # Need at least 4 EVs for a good sample
        selected_market = market
        break

if selected_market is None:
    # If no market has 4+ EVs, take the one with most EVs
    selected_market = max(positive_denom_markets, key=lambda x: x['num_evs'])

print(f"\n✅ SELECTED MARKET: {selected_market['market_id']}")
print(f"   Denominator: {selected_market['denominator']:.6f}")
print(f"   EV market share: {selected_market['s_ev']:.6f}")
print(f"   Number of EVs: {selected_market['num_evs']}")
print(f"   Total products: {selected_market['num_products']}")

# Now find models in this market and analyze their network effects
selected_market_id = selected_market['market_id']
market_df = df[df['market_ids'] == selected_market_id].copy()
ev_models = market_df[market_df['is_electric'] == 1]['model'].unique()

print(f"\n📋 EV MODELS IN SELECTED MARKET ({len(ev_models)} models):")
for model in ev_models:
    print(f"   - {model}")

# Calculate network effects for each model (difference between with and without network effects)
print(f"\n🔬 ANALYZING NETWORK EFFECTS FOR EACH MODEL:")
print("-" * 50)

# Get elasticity matrices for the selected market
elasticity_without = elasticity_matrices[selected_market_id]
elasticity_with = total_elasticity_matrices[selected_market_id]

# Get model names for this market
market_model_names = df[df['market_ids'] == selected_market_id]['model'].values

# Set proper indices for comparison
elasticity_without.index = market_model_names
elasticity_without.columns = market_model_names
elasticity_with.index = market_model_names  
elasticity_with.columns = market_model_names

# Use the same deduplication approach as in cell 18 for consistency
def remove_matrix_duplicates(matrix):
    """Remove duplicate rows and columns from matrix, keeping first occurrence"""
    # Remove duplicate rows
    matrix_no_dup_rows = matrix.loc[~matrix.index.duplicated(keep='first')]
    # Remove duplicate columns
    matrix_clean = matrix_no_dup_rows.loc[:, ~matrix_no_dup_rows.columns.duplicated(keep='first')]
    return matrix_clean

# Apply same deduplication method as used in cell 18
elasticity_without_clean = remove_matrix_duplicates(elasticity_without)
elasticity_with_clean = remove_matrix_duplicates(elasticity_with)

# Calculate network effects (difference in own-price elasticities)
network_effects = {}
for model in ev_models:
    if model in elasticity_without_clean.index:
        try:
            # Use the cleaned matrices to get diagonal values (consistent with cell 18)
            own_elast_without = elasticity_without_clean.loc[model, model]
            own_elast_with = elasticity_with_clean.loc[model, model]
            network_effect = own_elast_with - own_elast_without
            network_effects[model] = {
                'without_network': float(own_elast_without),
                'with_network': float(own_elast_with),
                'network_effect': float(network_effect)
            }
        except Exception as e:
            print(f"Error processing model {model}: {e}")
            continue

# Sort models by network effect (most negative first)
if network_effects:
    sorted_effects = sorted(network_effects.items(), key=lambda x: x[1]['network_effect'])
else:
    print("No network effects calculated successfully.")
    sorted_effects = []

print("Network effects on own-price elasticities:")
for model, effects in sorted_effects:
    print(f"  {model}:")
    print(f"    Without network: {effects['without_network']:.4f}")
    print(f"    With network:    {effects['with_network']:.4f}")
    print(f"    Network effect:  {effects['network_effect']:+.4f} {'(NEGATIVE)' if effects['network_effect'] < 0 else '(POSITIVE)'}")
    print()

# Select models with negative network effects
negative_effect_models = [model for model, effects in sorted_effects 
                         if effects['network_effect'] < 0]

print(f"🎯 MODELS WITH NEGATIVE NETWORK EFFECTS ({len(negative_effect_models)} found):")
for model in negative_effect_models:
    effect = network_effects[model]['network_effect']
    print(f"   - {model}: {effect:+.4f}")

# Select up to 4 models with most negative effects for the sample table
selected_models_for_table = negative_effect_models[:4]

if len(selected_models_for_table) < 4:
    print(f"\n⚠️  Only {len(selected_models_for_table)} models with negative effects found.")
    print("Adding models with least positive effects to reach 4 models...")
    
    positive_effect_models = [model for model, effects in sorted_effects 
                             if effects['network_effect'] >= 0]
    
    # Add models with smallest positive effects
    additional_needed = 4 - len(selected_models_for_table)
    selected_models_for_table.extend(positive_effect_models[:additional_needed])

print(f"\n🎉 FINAL SELECTION FOR SAMPLE TABLE:")
print(f"   Market: {selected_market_id}")
print(f"   Models: {selected_models_for_table}")

# Store the selection for the next cell
SELECTED_SAMPLE_MARKET = selected_market_id
SELECTED_SAMPLE_MODELS = selected_models_for_table

🔍 ANALYZING MARKETS AND MODELS FOR SAMPLE SELECTION
📊 Found 93 markets with positive denominators

Top 5 markets with highest positive denominators:
  1. P07Y2020: denominator=0.002241, s_ev=0.006004, EVs=64, products=189
  2. P07Y2021: denominator=0.002034, s_ev=0.008238, EVs=88, products=239
  3. P07Y2019: denominator=0.001791, s_ev=0.004112, EVs=40, products=144
  4. P04Y2020: denominator=0.001280, s_ev=0.004802, EVs=57, products=184
  5. P12Y2020: denominator=0.001239, s_ev=0.002650, EVs=76, products=207

✅ SELECTED MARKET: P07Y2020
   Denominator: 0.002241
   EV market share: 0.006004
   Number of EVs: 64
   Total products: 189

📋 EV MODELS IN SELECTED MARKET (62 models):
   - Soueast DX3
   - Jingyi S50
   - Fengguang E1
   - Yixuan
   - Toyota C-HR
   - Yudo π1
   - Yudo π3
   - Wuling Rongguang
   - Lingbao BOX
   - Hycan 007
   - Jiaji
   - Emgrand
   - Emgrand GSe
   - Xingyue
   - Binyue
   - Venucia D60
   - Venucia E30
   - Neta N01
   - Neta U
   - Audi A6L
   - Audi Q2L


In [47]:
# Patterns with market that have positive denominators
# ===============================================================================
# ANALYZE PATTERNS IN MARKETS WITH POSITIVE DENOMINATORS
# ===============================================================================

print("🔍 ANALYZING PATTERNS IN MARKETS WITH POSITIVE DENOMINATORS")
print("="*70)

# Filter markets with positive denominators
positive_markets = [info for info in feedback_analysis['market_info'] if info['denominator'] > 0]
negative_markets = [info for info in feedback_analysis['market_info'] if info['denominator'] <= 0]

print(f"📊 BASIC STATISTICS:")
print(f"   Total markets: {len(feedback_analysis['market_info'])}")
print(f"   Positive denominators: {len(positive_markets)} ({len(positive_markets)/len(feedback_analysis['market_info'])*100:.1f}%)")
print(f"   Negative/zero denominators: {len(negative_markets)} ({len(negative_markets)/len(feedback_analysis['market_info'])*100:.1f}%)")

# Convert to DataFrames for analysis
import pandas as pd
positive_df = pd.DataFrame(positive_markets)
negative_df = pd.DataFrame(negative_markets)
all_markets_df = pd.DataFrame(feedback_analysis['market_info'])

print(f"\n📈 COMPARISON OF MARKET CHARACTERISTICS:")
print("="*50)

# Compare key statistics
characteristics = ['s_ev', 'sum_gamma_ev', 'num_evs', 'num_products', 'denominator']

# Check if DataFrames are empty before calculating statistics
if len(positive_df) > 0 and len(negative_df) > 0:
    comparison_stats = pd.DataFrame({
        'Positive_Denom_Mean': positive_df[characteristics].mean(),
        'Negative_Denom_Mean': negative_df[characteristics].mean(),
        'Positive_Denom_Std': positive_df[characteristics].std(),
        'Negative_Denom_Std': negative_df[characteristics].std()
    })
    print(comparison_stats.round(6))
elif len(positive_df) > 0:
    print("Only positive denominator markets found:")
    print(positive_df[characteristics].describe())
elif len(negative_df) > 0:
    print("Only negative/zero denominator markets found:")
    print(negative_df[characteristics].describe())
else:
    print("No markets found for analysis!")

# Extract market information for detailed analysis
if len(positive_df) > 0:
    print(f"\n🎯 TOP 10 MARKETS WITH HIGHEST POSITIVE DENOMINATORS:")
    print("-"*60)
    top_positive = positive_df.nlargest(min(10, len(positive_df)), 'denominator')
    for i, row in top_positive.iterrows():
        print(f"{row['market_id']:>10}: denom={row['denominator']:>10.6f}, s_ev={row['s_ev']:>8.4f}, "
              f"EVs={row['num_evs']:>2}, products={row['num_products']:>2}")

if len(negative_df) > 0:
    print(f"\n🎯 TOP 10 MARKETS WITH MOST NEGATIVE DENOMINATORS:")
    print("-"*60)
    top_negative = negative_df.nsmallest(min(10, len(negative_df)), 'denominator')
    for i, row in top_negative.iterrows():
        print(f"{row['market_id']:>10}: denom={row['denominator']:>10.6f}, s_ev={row['s_ev']:>8.4f}, "
              f"EVs={row['num_evs']:>2}, products={row['num_products']:>2}")

# Analyze by province and year
print(f"\n📍 GEOGRAPHIC AND TEMPORAL PATTERNS:")
print("="*50)

# Extract province and year from market_ids (assuming format like 'P##Y####')
all_markets_df['province'] = all_markets_df['market_id'].str.extract(r'P(\d+)')[0]
all_markets_df['year'] = all_markets_df['market_id'].str.extract(r'Y(\d+)')[0]
all_markets_df['is_positive'] = all_markets_df['denominator'] > 0

# Province-level analysis
province_patterns = all_markets_df.groupby('province').agg({
    'is_positive': ['count', 'sum', 'mean'],
    'denominator': ['mean', 'std'],
    's_ev': 'mean',
    'num_evs': 'mean'
}).round(4)

province_patterns.columns = ['total_markets', 'positive_count', 'positive_rate', 
                           'avg_denominator', 'std_denominator', 'avg_s_ev', 'avg_num_evs']

print("PROVINCE PATTERNS (Top 10 by positive rate):")
print(province_patterns.nlargest(10, 'positive_rate'))

# Year-level analysis
year_patterns = all_markets_df.groupby('year').agg({
    'is_positive': ['count', 'sum', 'mean'],
    'denominator': ['mean', 'std'],
    's_ev': 'mean',
    'num_evs': 'mean'
}).round(4)

year_patterns.columns = ['total_markets', 'positive_count', 'positive_rate', 
                        'avg_denominator', 'std_denominator', 'avg_s_ev', 'avg_num_evs']

print(f"\nYEAR PATTERNS:")
print(year_patterns)

# Correlation analysis
print(f"\n📊 CORRELATION ANALYSIS:")
print("="*50)
corr_vars = ['denominator', 's_ev', 'sum_gamma_ev', 'num_evs', 'num_products']
correlation_matrix = all_markets_df[corr_vars].corr()
print("Correlation with denominator:")
print(correlation_matrix['denominator'].sort_values(ascending=False))

# Threshold analysis - what EV share levels lead to positive denominators?
print(f"\n🎚️ EV SHARE THRESHOLD ANALYSIS:")
print("="*50)
ev_share_bins = pd.cut(all_markets_df['s_ev'], bins=10, precision=4)
threshold_analysis = all_markets_df.groupby(ev_share_bins).agg({
    'is_positive': ['count', 'sum', 'mean'],
    'denominator': 'mean'
}).round(4)
threshold_analysis.columns = ['total', 'positive_count', 'positive_rate', 'avg_denominator']
print("EV Share ranges and positive denominator rates:")
print(threshold_analysis)

print(f"\n✅ PATTERN ANALYSIS COMPLETE")

🔍 ANALYZING PATTERNS IN MARKETS WITH POSITIVE DENOMINATORS
📊 BASIC STATISTICS:
   Total markets: 155
   Positive denominators: 93 (60.0%)
   Negative/zero denominators: 62 (40.0%)

📈 COMPARISON OF MARKET CHARACTERISTICS:
              Positive_Denom_Mean  Negative_Denom_Mean  Positive_Denom_Std  \
s_ev                     0.001654             0.005343            0.001836   
sum_gamma_ev             0.002063             0.011309            0.002759   
num_evs                 53.849462           126.580645           25.822953   
num_products           181.731183           268.451613           40.473593   
denominator              0.000425            -0.001392            0.000407   

              Negative_Denom_Std  
s_ev                    0.003895  
sum_gamma_ev            0.008963  
num_evs                42.558561  
num_products           52.421582  
denominator             0.001610  

🎯 TOP 10 MARKETS WITH HIGHEST POSITIVE DENOMINATORS:
----------------------------------------------

C:\Users\Lenovo\AppData\Local\Temp\ipykernel_16660\3154021554.py:114: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  threshold_analysis = all_markets_df.groupby(ev_share_bins).agg({


In [48]:
# Generate sample tables for selected EV models with 6 digits, showing both derivatives and elasticities
# Using market with positive denominators and models with negative network effects

sample_market = 'P07Y2021'  # Market with positive denominator
selected_models = ['XPeng P5', 'BYD D1', 'Toyota C-HR', 'Kia K3', 'Qin Pro']  # Models with negative network effects

print(f"📊 GENERATING SAMPLE TABLES")
print(f"Market: {sample_market}")
print(f"Selected models: {selected_models}")
print()

# Ensure only unique model names and present in the selected market
market_df = df[(df['market_ids'] == sample_market) & (df['is_electric'] == 1)]
available_models = [m for m in selected_models if m in market_df['model'].unique()]

print(f"Available models in market: {available_models}")
print()

# Get matrices for the sample market
derivative_matrix_full = derivative_matrices[sample_market]
elasticity_matrix_full = elasticity_matrices[sample_market]
total_elasticity_matrix_full = total_elasticity_matrices[sample_market]

model_names = df[df['market_ids'] == sample_market]['model'].values

# Set index and columns to model names
derivative_matrix_full.index = model_names
derivative_matrix_full.columns = model_names
elasticity_matrix_full.index = model_names
elasticity_matrix_full.columns = model_names
total_elasticity_matrix_full.index = model_names
total_elasticity_matrix_full.columns = model_names

# Create a mask for first occurrence of each model name
_, idx = np.unique(model_names, return_index=True)
unique_model_names = model_names[np.sort(idx)]

# Filter matrices to only include first occurrence for each model name
derivative_matrix = derivative_matrix_full.loc[unique_model_names, unique_model_names]
elasticity_matrix = elasticity_matrix_full.loc[unique_model_names, unique_model_names]
total_elasticity_matrix = total_elasticity_matrix_full.loc[unique_model_names, unique_model_names]

# Use available models as selected (don't filter for uniqueness in model selection)
print(f"Selected models for tables: {available_models}")

# Sample matrices for selected models
sample_derivatives = derivative_matrix.loc[available_models, available_models]
sample_elasticities = elasticity_matrix.loc[available_models, available_models]
sample_total_elasticities = total_elasticity_matrix.loc[available_models, available_models]

# Also get total derivatives with network effects
sample_total_deriv_df = pd.DataFrame(
    total_derivatives_dict[sample_market],
    index=derivative_matrix.index,
    columns=derivative_matrix.columns
).loc[available_models, available_models]

# Remove duplicate rows and columns from the matrices (keep first occurrence)
def remove_matrix_duplicates(matrix):
    """Remove duplicate rows and columns from matrix, keeping first occurrence"""
    # Remove duplicate rows
    matrix_no_dup_rows = matrix.loc[~matrix.index.duplicated(keep='first')]
    # Remove duplicate columns
    matrix_clean = matrix_no_dup_rows.loc[:, ~matrix_no_dup_rows.columns.duplicated(keep='first')]
    return matrix_clean

# Clean all sample matrices to remove duplicates
sample_derivatives = remove_matrix_duplicates(sample_derivatives)
sample_elasticities = remove_matrix_duplicates(sample_elasticities)
sample_total_elasticities = remove_matrix_duplicates(sample_total_elasticities)
sample_total_deriv_df = remove_matrix_duplicates(sample_total_deriv_df)

def matrix_to_latex(matrix, caption):
    latex = "\\begin{table}[!htbp]\n\\centering\n"
    latex += "\\begin{tabular}{@{\\extracolsep{5pt}}l" + "c" * len(matrix.columns) + "}\n"
    latex += "\\\\[-1.8ex]\\hline\n\\\\hline \\\\[-1.8ex]\n"
    latex += "& " + " & ".join(matrix.columns) + " \\\\\n"
    latex += "\\hline \\\\[-1.8ex]\n"
    for idx, row in matrix.iterrows():
        latex += f"{idx} & " + " & ".join([f'{v:.6f}' for v in row]) + " \\\\\n"
    latex += "\\hline \\\\[-1.8ex]\n"
    latex += "\\end{tabular}\n"
    latex += f"\\caption{{{caption}}}\n"
    latex += "\\end{table}\n"
    return latex

# Generate LaTeX tables
latex_derivatives = matrix_to_latex(sample_derivatives, "Derivative matrix (∂s_j/∂p_k) - EV models with negative network effects")
latex_elasticities_no_network = matrix_to_latex(sample_elasticities, "Elasticity matrix WITHOUT network effects - EV models with negative network effects")
latex_elasticities_with_network = matrix_to_latex(sample_total_elasticities, "Elasticity matrix WITH network effects - EV models with negative network effects")

# Save tables
with open('GraphsTables/sample_derivatives_selected.tex', 'w', encoding='utf-8') as f:
    f.write(latex_derivatives)

with open('GraphsTables/sample_elasticity_selected_without_network.tex', 'w', encoding='utf-8') as f:
    f.write(latex_elasticities_no_network)

with open('GraphsTables/sample_elasticity_selected_with_network.tex', 'w', encoding='utf-8') as f:
    f.write(latex_elasticities_with_network)

# Display results
print("\nELASTICITIES WITHOUT NETWORK EFFECTS:")
print("=" * 50)
print(sample_elasticities)
print("\nELASTICITIES WITH NETWORK EFFECTS:")
print("=" * 50)
print(sample_total_elasticities)

# Show the network effects for verification using the same cleaned matrices
print("\n🔍 NETWORK EFFECTS VERIFICATION:")
print("=" * 50)
for model in sample_derivatives.index:
    # Get diagonal values from the cleaned matrices (same as displayed in tables)
    own_elast_without_cleaned = sample_elasticities.loc[model, model]
    own_elast_with_cleaned = sample_total_elasticities.loc[model, model]
    network_effect_cleaned = own_elast_with_cleaned - own_elast_without_cleaned
    
    print(f"{model}:")
    print(f"  Own-price elasticity without network: {own_elast_without_cleaned:.4f}")
    print(f"  Own-price elasticity with network:    {own_elast_with_cleaned:.4f}")
    print(f"  Network effect:                       {network_effect_cleaned:+.4f}")
    print()

📊 GENERATING SAMPLE TABLES
Market: P07Y2021
Selected models: ['XPeng P5', 'BYD D1', 'Toyota C-HR', 'Kia K3', 'Qin Pro']

Available models in market: ['XPeng P5', 'BYD D1', 'Toyota C-HR', 'Kia K3', 'Qin Pro']

Selected models for tables: ['XPeng P5', 'BYD D1', 'Toyota C-HR', 'Kia K3', 'Qin Pro']

ELASTICITIES WITHOUT NETWORK EFFECTS:
             XPeng P5    BYD D1  Toyota C-HR    Kia K3   Qin Pro
XPeng P5    -8.497132  0.081654     0.004500  0.000036  0.000006
BYD D1       0.025155 -8.405643     0.004500  0.000036  0.000006
Toyota C-HR  0.025155  0.081654    -8.599428  0.000036  0.000006
Kia K3       0.000048  0.000156     0.000009 -6.943025  0.003050
Qin Pro      0.000048  0.000156     0.000009  0.018988 -4.798381

ELASTICITIES WITH NETWORK EFFECTS:
             XPeng P5    BYD D1  Toyota C-HR    Kia K3   Qin Pro
XPeng P5    -8.667018 -0.469814    -0.025894 -0.159037 -0.025542
BYD D1      -0.143595 -8.953423    -0.025691 -0.157973 -0.025371
Toyota C-HR -0.145145 -0.471157    -8.629897